# PoliMillionaire - TF-IDF retrieval, no LLM

Runs all competitions through the API using the local SimpleWiki TF-IDF index. Each competition is played 5 times and every question attempt is printed and saved to CSV.

In [1]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "api_client" / "NLP_assignment_api_client").exists() and (path / "project" / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()
CLIENT_PARENT = PROJECT_ROOT / "api_client" / "NLP_assignment_api_client"
SRC_DIR = PROJECT_ROOT / "project" / "src"

for path in [str(CLIENT_PARENT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Tommaso\Code\NLP_polimi_26


In [2]:
from getpass import getpass
from millionaire_client import MillionaireClient
from retrieval_quiz_runner import load_retrieval_index, run_all_competitions, summarize, write_logs

API_URL = os.getenv("POLIMILLIONAIRE_API_URL", "http://131.175.15.22:51111/")
USERNAME = os.getenv("POLIMILLIONAIRE_USERNAME") or input("Username: ").strip()
PASSWORD = os.getenv("POLIMILLIONAIRE_PASSWORD") or getpass("Password: ")

client = MillionaireClient(API_URL, timeout=30)
user = client.login(USERNAME, PASSWORD)
print(f"Logged in as {user.username} ({user.role})")

Logged in as NeuroniNegroni (student)


In [3]:
INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "simplewiki_160w_tfidf.joblib"
if not INDEX_PATH.exists():
    raise FileNotFoundError(f"Missing index: {INDEX_PATH}")

index = load_retrieval_index(INDEX_PATH)
print("Loaded index:", INDEX_PATH)
print("Kind:", index["kind"])
print("Chunks:", len(index["docs"]))

Loaded index: C:\Users\Tommaso\Code\NLP_polimi_26\data\indexes\simplewiki_160w_tfidf.joblib
Kind: tfidf
Chunks: 434093


In [4]:
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"{comp.id}: {comp.name} | max levels: {comp.max_levels} | questions: {comp.question_count}")

0: Entertainment | max levels: 15 | questions: 843
1: Ancient History and Politics | max levels: 15 | questions: 456
2: Science and Nature | max levels: 15 | questions: 5395
3: Maths | max levels: 15 | questions: 390


In [7]:
ATTEMPTS_PER_COMPETITION = 10
STRATEGY_NAME = "simplewiki_tfidf_no_llm"

rows = run_all_competitions(
    client=client,
    index=index,
    strategy_name=STRATEGY_NAME,
    attempts_per_competition=ATTEMPTS_PER_COMPETITION,
)

output_path = PROJECT_ROOT / "logs" / "simplewiki_tfidf_no_llm_all_competitions.csv"
write_logs(rows, output_path)
print(f"Wrote {len(rows)} rows to {output_path}")

[simplewiki_tfidf_no_llm] comp=0 Entertainment attempt=1 q=1 level=0 chosen=0 correct=False earned=0 latency=0.5997911999875214
Q: What is the primary reason Thanos proposes to kill half of Titan's population?
A: To gain power over his people
Top evidence: Nenets Autonomous Okrug :: 18.6% of the population, the Komi people were 9.0% of the population and other ethnicities were 6.3% of the population. In 1989, the population was at 54,840 people, but a rapid decrease in population occurred shortly afterwards. The population has seen a slow increase since 2002. References Other websites * Official website

[simplewiki_tfidf_no_llm] comp=0 Entertainment attempt=2 q=1 level=0 chosen=3 correct=False earned=0 latency=0.5854383000405505
Q: What was the significance of the 1977 film 'Smokey and the Bandit' to Quentin Tarantino?
A: It was the first film he saw in a theater
Top evidence: Smokey and the Bandit Part 3 :: Smokey and the Bandit Part 3 is a 1983 American action comedy movie and a spi

In [8]:
summarize(rows)


Summary
0 Entertainment: rows=21, sessions=10, correct=11, row_acc=0.524, max_seen_level=0, avg_latency=0.606s
1 Ancient History and Politics: rows=14, sessions=10, correct=4, row_acc=0.286, max_seen_level=0, avg_latency=0.596s
2 Science and Nature: rows=16, sessions=10, correct=6, row_acc=0.375, max_seen_level=0, avg_latency=0.586s
3 Maths: rows=15, sessions=10, correct=5, row_acc=0.333, max_seen_level=0, avg_latency=0.603s
